In [1]:
import valenspy
from valenspy._utilities import load_yml, create_named_regex
import pandas as pd

DATASET_PATHS = load_yml("dataset_PATHS")

/dodrio/scratch/projects/2022_200/project_output/RMIB-UGent/vsc46032_kobe/conda_envs/valenspy_env/lib/python3.11/site-packages/esmpy/interface/loadESMF.py:94: VersionWarning: ESMF installation version 8.8.0, ESMPy version 8.8.0b0
  warnings.warn("ESMF installation version {}, ESMPy version {}".format(


In [2]:
m = valenspy.InputManager("hortense") #Takes 1 minute to load - ideally cache this somehow

/dodrio/scratch/projects/2022_200/project_output/RMIB-UGent/vsc46032_kobe/ValEnsPy/src/valenspy/input/manager.py:169: UserWarning: Dataset ERA5 is missing the required identifier 'period' for filtering in its pattern or metadata.
  warnings.warn(f"Dataset {dataset_name} is missing the required identifier '{identifier}' for filtering in its pattern or metadata.")
/dodrio/scratch/projects/2022_200/project_output/RMIB-UGent/vsc46032_kobe/ValEnsPy/src/valenspy/input/manager.py:169: UserWarning: Dataset ERA5-Land is missing the required identifier 'period' for filtering in its pattern or metadata.
  warnings.warn(f"Dataset {dataset_name} is missing the required identifier '{identifier}' for filtering in its pattern or metadata.")
/dodrio/scratch/projects/2022_200/project_output/RMIB-UGent/vsc46032_kobe/ValEnsPy/src/valenspy/input/manager.py:169: UserWarning: Dataset EOBS is missing the required identifier 'period' for filtering in its pattern or metadata.
  warnings.warn(f"Dataset {dataset_

In [16]:
m.readable_catalog

variable_id
source_id frequency             
CCLM      ALB_RAD          18672
          ALHFL_PL           129
          ALHFL_S            385
          ASHFL_S          17902
          ASOB_S             385
...                          ...
ERA5-Land daily             3699
          hourly            1899
          monthly           1257
MAR       daily               44
RADCLIM   hourly            2186

[70 rows x 1 columns]

In [3]:
from intake_esm.cat import Aggregation

In [4]:
agg_list = [
    Aggregation(type="union", attribute_name="variable_id"), 
    Aggregation(type="join_existing", 
                attribute_name="time_range", 
                options={
                    "dim": "time",
                    "coords": "minimal",
                    "compat": "override"
                }
                ),
]

In [5]:
m.create_intake_esm_json_catalog(output_path=".", aggregations=agg_list, groupby_attrs=["source_id"])

In [6]:
m.df.columns

Index(['activity_id', 'resolution', 'version', 'experiment_id', 'domain_id',
       'variable_id', 'frequency', 'variable_id2', 'domain_id_2',
       'frequency_2', 'year', 'path', 'source_id', 'start_year', 'end_year',
       'time_range', 'raw_variable_id', 'aggregation', 'resolution2',
       'domain_id2', 'yearmonthday', 'driving_source_id', 'regridding_method'],
      dtype='object')

In [7]:
m.cat.aggregation_control

AggregationControl(variable_column_name='variable_id', groupby_attrs=['source_id'], aggregations=[Aggregation(type=<AggregationType.union: 'union'>, attribute_name='variable_id', options={}), Aggregation(type=<AggregationType.join_existing: 'join_existing'>, attribute_name='time_range', options={'dim': 'time', 'coords': 'minimal', 'compat': 'override'})])

In [8]:
m.cat.df.columns

Index(['activity_id', 'resolution', 'version', 'experiment_id', 'domain_id',
       'variable_id', 'frequency', 'variable_id2', 'domain_id_2',
       'frequency_2', 'year', 'path', 'source_id', 'start_year', 'end_year',
       'time_range', 'raw_variable_id', 'aggregation', 'resolution2',
       'domain_id2', 'yearmonthday', 'driving_source_id', 'regridding_method'],
      dtype='object')

In [3]:
#Test adding a user defined dataset to the catalog not yet in the package

alaro_data_root = "/dodrio/scratch/projects/2022_200/project_output/RMIB-UGent/CORDEXbeII/run_ALARO_sfx/out/PROD_ERA5_EUR11/19780101"
pattern = "<project_id>/<activity_id>/<domain_id>/<institution_id>/<driving_source_id>/<driving_experiment_id>/<driving_variant_label>/<source_id>/<version_realization>/<frequency>/<variable_id>/<version>/<variable_id_2>_<domain_id_2>_<driving_source_id_2>_<driving_experiment_id_2>_<driving_variant_label_2>_<institution_id_2>_<source_id_2>_<version_realization_2>_<frequency_2>_<time_range>.nc"

m.update_catalog("ALARO", alaro_data_root, pattern)

/dodrio/scratch/projects/2022_200/project_output/RMIB-UGent/vsc46032_kobe/ValEnsPy/src/valenspy/input/manager.py:76: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.DataFrame(data)


In [4]:
variables = ["pr","tas","tasmin","tasmax"]
ds_al = m.open_dataset("ALARO", variables=variables, freq="day")

In [5]:
ds_mar = m.open_dataset(dataset_name="MAR", variables=variables, freq="daily")

Variable metadata is missing or incorrect
The file is NOT ValEnsPy CF compliant.
78.57% of the variables are ValEnsPy CF compliant
ValEnsPy CF compliant: ['tas', 'huss', 'hurs', 'uas', 'vas', 'rsds', 'clt', 'tasmax', 'tasmin', 'prsn', 'pr']
Unknown to ValEnsPy: ['TIME_bnds', 'ZUVLEV_bnds', 'RHZmin']


In [9]:
from intake_esm import esm_datastore
from intake_esm.cat import ESMCatalogModel
print(type(m.cat))
print(isinstance(m.cat, ESMCatalogModel))

<class 'intake_esm.cat.ESMCatalogModel'>
True


In [13]:
m.cat.model_dump()

{'esmcat_version': '0.1.0',
 'attributes': [{'column_name': 'activity_id', 'vocabulary': ''},
  {'column_name': 'resolution', 'vocabulary': ''},
  {'column_name': 'version', 'vocabulary': ''},
  {'column_name': 'experiment_id', 'vocabulary': ''},
  {'column_name': 'domain_id', 'vocabulary': ''},
  {'column_name': 'variable_id', 'vocabulary': ''},
  {'column_name': 'frequency', 'vocabulary': ''},
  {'column_name': 'variable_id2', 'vocabulary': ''},
  {'column_name': 'domain_id_2', 'vocabulary': ''},
  {'column_name': 'frequency_2', 'vocabulary': ''},
  {'column_name': 'year', 'vocabulary': ''},
  {'column_name': 'path', 'vocabulary': ''},
  {'column_name': 'source_id', 'vocabulary': ''},
  {'column_name': 'start_year', 'vocabulary': ''},
  {'column_name': 'end_year', 'vocabulary': ''},
  {'column_name': 'time_range', 'vocabulary': ''},
  {'column_name': 'raw_variable_id', 'vocabulary': ''},
  {'column_name': 'aggregation', 'vocabulary': ''},
  {'column_name': 'resolution2', 'vocabulary'

In [14]:
esm_datastore(m.cat.model_dump())

KeyError: 'esmcat'

In [4]:
m.readable_catalog

variable_id
source_id frequency             
CCLM      ALB_RAD          18672
          ALHFL_PL           129
          ALHFL_S            385
          ASHFL_S          17902
          ASOB_S             385
...                          ...
ERA5-Land daily             3699
          hourly            1899
          monthly           1257
MAR       daily               44
RADCLIM   hourly            2186

[70 rows x 1 columns]

In [29]:
cat_subset = m.cat.search(query={"source_id":"MAR"})
cat_subset

,activity_id,resolution,version,experiment_id,domain_id,variable_id,frequency,variable_id2,domain_id_2,frequency_2,...,start_year,end_year,time_range,raw_variable_id,aggregation,resolution2,domain_id2,yearmonthday,driving_source_id,regridding_method
0,RCM,~5km,v3.14,evaluation,Belgium,"[prsn, uas, vas, tas, rsds, huss, tasmin, pr, ...",daily,NaN,NaN,NaN,...,1985,1985,"[1985-01-01 00:00:00, 1985-12-31 00:00:00]","[RHZ, TTZmax, SWD, MBRR, MBSF, TTZmin, TTZ, V2...",NaN,NaN,NaN,NaN,ERA5,remapbil
1,RCM,~5km,v3.14,evaluation,Belgium,"[prsn, uas, vas, tas, rsds, huss, tasmin, pr, ...",daily,NaN,NaN,NaN,...,1995,1995,"[1995-01-01 00:00:00, 1995-12-31 00:00:00]","[RHZ, TTZmax, SWD, MBRR, MBSF, TTZmin, TTZ, V2...",NaN,NaN,NaN,NaN,ERA5,remapbil
2,RCM,~5km,v3.14,evaluation,Belgium,"[prsn, uas, vas, tas, rsds, huss, tasmin, pr, ...",daily,NaN,NaN,NaN,...,2015,2015,"[2015-01-01 00:00:00, 2015-12-31 00:00:00]","[RHZ, TTZmax, SWD, MBRR, MBSF, TTZmin, TTZ, V2...",NaN,NaN,NaN,NaN,ERA5,remapbil
3,RCM,~5km,v3.14,evaluation,Belgium,"[prsn, uas, vas, tas, rsds, huss, tasmin, pr, ...",daily,NaN,NaN,NaN,...,1998,1998,"[1998-01-01 00:00:00, 1998-12-31 00:00:00]","[RHZ, TTZmax, SWD, MBRR, MBSF, TTZmin, TTZ, V2...",NaN,NaN,NaN,NaN,ERA5,remapbil
4,RCM,~5km,v3.14,evaluation,Belgium,"[prsn, uas, vas, tas, rsds, huss, tasmin, pr, ...",daily,NaN,NaN,NaN,...,2023,2023,"[2023-01-01 00:00:00, 2023-12-31 00:00:00]","[RHZ, TTZmax, SWD, MBRR, MBSF, TTZmin, TTZ, V2...",NaN,NaN,NaN,NaN,ERA5,remapbil
5,RCM,~5km,v3.14,evaluation,Belgium,"[prsn, uas, vas, tas, rsds, huss, tasmin, pr, ...",daily,NaN,NaN,NaN,...,2014,2014,"[2014-01-01 00:00:00, 2014-12-31 00:00:00]","[RHZ, TTZmax, SWD, MBRR, MBSF, TTZmin, TTZ, V2...",NaN,NaN,NaN,NaN,ERA5,remapbil
6,RCM,~5km,v3.14,evaluation,Belgium,"[prsn, uas, vas, tas, rsds, huss, tasmin, pr, ...",daily,NaN,NaN,NaN,...,2005,2005,"[2005-01-01 00:00:00, 2005-12-31 00:00:00]","[RHZ, TTZmax, SWD, MBRR, MBSF, TTZmin, TTZ, V2...",NaN,NaN,NaN,NaN,ERA5,remapbil
7,RCM,~5km,v3.14,evaluation,Belgium,"[prsn, uas, vas, tas, rsds, huss, tasmin, pr, ...",daily,NaN,NaN,NaN,...,2001,2001,"[2001-01-01 00:00:00, 2001-12-31 00:00:00]","[RHZ, TTZmax, SWD, MBRR, MBSF, TTZmin, TTZ, V2...",NaN,NaN,NaN,NaN,ERA5,remapbil
8,RCM,~5km,v3.14,evaluation,Belgium,"[prsn, uas, vas, tas, rsds, huss, tasmin, pr, ...",daily,NaN,NaN,NaN,...,2018,2018,"[2018-01-01 00:00:00, 2018-12-31 00:00:00]","[RHZ, TTZmax, SWD, MBRR, MBSF, TTZmin, TTZ, V2...",NaN,NaN,NaN,NaN,ERA5,remapbil
9,RCM,~5km,v3.14,evaluation,Belgium,"[prsn, uas, vas, tas, rsds, huss, tasmin, pr, ...",daily,NaN,NaN,NaN,...,1984,1984,"[1984-01-01 00:00:00, 1984-12-31 00:00:00]","[RHZ, TTZmax, SWD, MBRR, MBSF, TTZmin, TTZ, V2...",NaN,NaN,NaN,NaN,ERA5,remapbil


In [30]:
dset_dict = m.cat.to_dataset_dict(
    xarray_open_kwargs={"consolidated": True, "decode_times": True, "use_cftime": True, "chunks": "auto"}
)


AttributeError: 'ESMCatalogModel' object has no attribute 'to_dataset_dict'

In [32]:
from intake_esm.cat import ESMCatalogModel

In [ ]:
[key for key in dset_dict.keys()]

In [34]:
import intake_esm
url = intake_esm.tutorial.get_url('google_cmip6')
print(url)

https://raw.githubusercontent.com/intake/intake-esm/main/tutorial-catalogs/GOOGLE-CMIP6.json


In [39]:
#Save the catalog to a file
m.cat.save("catalog.yml")

TypeError: Object of type PosixPath is not JSON serializable

In [37]:
import intake

cat = intake.open_esm_datastore(m.cat.json())
cat

/tmp/ipykernel_3915163/3149882370.py:2: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  cat = intake.open_esm_datastore(m.cat.json())


OSError: [Errno 36] File name too long: '/dodrio/scratch/projects/2022_200/project_output/RMIB-UGent/vsc46032_kobe/ValEnsPy/examples/{"esmcat_version":"0.1.0","attributes":[{"column_name":"activity_id","vocabulary":""},{"column_name":"resolution","vocabulary":""},{"column_name":"version","vocabulary":""},{"column_name":"experiment_id","vocabulary":""},{"column_name":"domain_id","vocabulary":""},{"column_name":"variable_id","vocabulary":""},{"column_name":"frequency","vocabulary":""},{"column_name":"variable_id2","vocabulary":""},{"column_name":"domain_id_2","vocabulary":""},{"column_name":"frequency_2","vocabulary":""},{"column_name":"year","vocabulary":""},{"column_name":"path","vocabulary":""},{"column_name":"source_id","vocabulary":""},{"column_name":"start_year","vocabulary":""},{"column_name":"end_year","vocabulary":""},{"column_name":"time_range","vocabulary":""},{"column_name":"raw_variable_id","vocabulary":""},{"column_name":"aggregation","vocabulary":""},{"column_name":"resolu